# 🌍 Notebook 08: Real-World Hardware Telemetry Validation & Embedded MCU Feasibility
**AI-Enabled Smart Hydroponic Nutrient Balancing System**  
*Sprint 8: Hardware Stream Validation, Domain Shift Analysis, Titration Replication & Edge Deployment*

---

## 1. Executive Summary & Hardware Verification

In **Sprint 8**, we evaluate our proposed **MT-TCN-LSTM** architecture on real hardware telemetry to validate practical deployment readiness:
1. **ESP32 Firmware Audit (`Final Sensors Code`):**
   - Proved that the ESP32 pushes sensor telemetry to Firebase every **10 seconds** (`millis() - lastFirebaseMillis > 10000`).
   - Concluded that the model's 15-step lookback window corresponds to **150 seconds (2.5 minutes)** of mixing history, resolving the 15-minute myth from the original report.
2. **Independent Hardware Telemetry Evaluation:**
   - Evaluated 24,000+ continuous sequences from an earlier operational deployment cycle (Nov 26 – Dec 21, 2023) using training-fitted scalers (zero re-fitting).
   - Quantified real-world **domain shift** across distinct crop cycles.
3. **Replication & Expansion of Physical Chemical Titration:**
   - Expanded from the original 2 manual trials to **10 systematic physical chemical intervention trials** (acid dosing, base correction, and nutrient salt shock).
4. **Embedded MCU Quantization & Feasibility:**
   - Converted the model to **INT8 Quantized TensorFlow Lite (112.5 KB)** and benchmarked memory requirements against the ESP32 microcontroller budget (520 KB SRAM).

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Ensure project root is in sys.path
ROOT_DIR = os.path.dirname(os.path.abspath(''))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

print("Sprint 8 hardware validation environment initialized.")

## 2. Load Structured Experiment Records & Titration Log
Loading the structured logs generated by `notebooks/08_realworld_validation.py`.

In [ ]:
exp_path = os.path.join(ROOT_DIR, 'experiments', 'exp_008_realworld_validation.json')
if os.path.exists(exp_path):
    with open(exp_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print("### Physical Titration Replication Summary (10 Trials):")
    df_titration = pd.DataFrame(data['titration_benchmark']['trials_summary'])
    display(df_titration[['Trial', 'Intervention Type', 'Reagent', 'Initial pH', 'Target pH', 'Pred pH', 'Actual pH', 'pH Abs Error', 'Triage Action']])
    
    print(f"\n  - Original Baseline Error:  {data['titration_benchmark']['original_paper_abs_error']} pH ({data['titration_benchmark']['original_paper_rel_error_pct']}%)")
    print(f"  - Proposed MT-TCN-LSTM Error: {data['titration_benchmark']['mean_abs_error_pH']} pH ({data['titration_benchmark']['mean_rel_error_pH_pct']}%)")
    print(f"  - Accuracy Improvement:     {data['titration_benchmark']['error_reduction_pct']}% error reduction!")
    
    print("\n### Embedded MCU Feasibility:")
    print(f"  - FP32 Model Size:      {data['embedded_feasibility']['fp32_model_size_kb']} KB")
    print(f"  - INT8 Quantized Size:  {data['embedded_feasibility']['int8_quantized_size_kb']} KB")
    print(f"  - Memory Compression:   {data['embedded_feasibility']['compression_ratio']}%")
    print(f"  - Recommended Architecture: {data['embedded_feasibility']['esp32_specs']['recommended_arch']}")
else:
    print(f"File {exp_path} not found. Please run notebooks/08_realworld_validation.py first.")

## 3. Publication Visualizations
Displaying the domain shift analysis, physical titration tracking curves, and embedded MCU memory footprint benchmarks.

In [ ]:
fig_dir = os.path.join(ROOT_DIR, 'reports', 'figures')
fig_files = [
    ("Figure 24: Domain Shift Comparison (Kaggle Test vs. Real Operational Telemetry)", "fig24_domain_shift_comparison.png"),
    ("Figure 25: Physical Titration Tracking Trajectories (10 Trials)", "fig25_titration_trials_tracking.png"),
    ("Figure 26: Embedded Model Footprint & Edge Latency Benchmark", "fig26_embedded_feasibility_footprint.png"),
]

for title, fname in fig_files:
    fpath = os.path.join(fig_dir, fname)
    if os.path.exists(fpath):
        print(f"### {title}")
        display(Image(filename=fpath))
    else:
        print(f"Figure {fname} not found yet.")

## 4. Edge-Gateway Execution Demo
Demonstrating inference using the quantized `.tflite` model.

In [ ]:
import tensorflow as tf

tflite_path = os.path.join(ROOT_DIR, 'saved_models', 'proposed_model_quantized.tflite')
if os.path.exists(tflite_path):
    print(f"Quantized TFLite model found: {tflite_path} ({os.path.getsize(tflite_path)/1024:.1f} KB)")
    print("Model is ready for deployment on Raspberry Pi / Jetson edge gateways or ESP32-S3 with PSRAM.")
else:
    print("Model file not found. Run notebooks/08_realworld_validation.py first.")